In [6]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
import cvxpy as cp
import os
from google.colab import drive

# ==========================================
# 1. MOUNT GOOGLE DRIVE
# ==========================================
# Bước này sẽ yêu cầu bạn cấp quyền truy cập Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
!pip install ecos
!pip install scs

In [8]:
# --- CES / Linear (m = CES exponent, m=1 → Linear, m='inf' → Leontief-like max) ---
def calculate_ces_utility(allocation_vec, valuation_vec, m_ces = 1/2):
    eps = 1e-9
    return np.power(np.power(allocation_vec + eps, m_ces).T @ valuation_vec, (1/m_ces))


# --- COBB–DOUGLAS UTILITY ---
def calculate_cd_utility(allocation_vec, valuations_vec):
    eps = 1e-12
    v = np.atleast_2d(valuations_vec)
    x = np.atleast_2d(allocation_vec)

    # Normalize weights α_j = v_j / sum(v)
    weights = v / (np.sum(v, axis=1, keepdims=True))

    # log utility = Σ α_j log(x_j)
    log_u = np.sum(weights * np.log(x + eps), axis=1)

    u = np.exp(log_u)

    return u


# --- LEONTIEF UTILITY ---
def calculate_leo_utility(allocation_vec, valuations_vec):
    v = np.atleast_2d(valuations_vec)
    x = np.atleast_2d(allocation_vec)
    # 1. Khởi tạo mảng kết quả là Vô cực (Infinity)
    # Để nếu v=0, giá trị tại đó vẫn là Inf và không bị hàm min chọn phải
    ratios = np.full_like(x, 1e30)

    # 2. Chỉ thực hiện phép chia ở những nơi v > 0
    # Ghi đè kết quả phép chia vào mảng ratios
    np.divide(x, v, out=ratios, where=(v > 1e-12))

    # 3. Lấy Min theo hàng (axis=1)
    u = np.min(ratios, axis=1)

    return u


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import os
import time

# ==============================================================================
# 1. CÁC HÀM CỐT LÕI (CORE FUNCTIONS)
# ==============================================================================

def build_A_matrix(n_goods, n_slots):
    """Tạo ma trận ánh xạ Goods -> Slots"""
    A = np.zeros((n_slots, n_goods))
    for j in range(n_goods):
        A[j % n_slots, j] = 1.0
    return A

import numpy as np

def buyer_best_response_cvx(v_i, p, A, q_i, B_i, utility_type="Linear"):
    """
    Phiên bản siêu tốc của buyer_best_response.
    Sử dụng Closed-form solution cho mọi hàm utility.
    KHÔNG DÙNG CVXPY.
    """
    # 1. Tính giá hiệu dụng (Effective Price)
    # p: (M,), A.T @ q_i: (M,)
    effective_price = p + (A.T @ q_i)

    # An toàn: Đảm bảo giá > 0 để tránh chia cho 0
    safe_price = np.maximum(effective_price, 1e-9)

    m_goods = len(v_i)
    x = np.zeros(m_goods)

    # =========================================================
    # 1. LINEAR UTILITY: U = sum(v * x)
    # =========================================================
    if utility_type == "Linear":
        # Chiến thuật: Bang-per-buck (Mua tất tay món hời nhất)
        bang_per_buck = v_i / safe_price
        best_idx = np.argmax(bang_per_buck)
        x[best_idx] = B_i / safe_price[best_idx]

    # =========================================================
    # 2. COBB-DOUGLAS: U = prod(x^alpha) hoặc sum(alpha * log(x))
    # =========================================================
    elif utility_type == "Cobb-Douglas":
        # Chuẩn hóa alpha (Bắt buộc để thỏa mãn Budget constraint)
        sum_v = np.sum(v_i)
        if sum_v > 0:
            alpha = v_i / sum_v
        else:
            alpha = np.zeros_like(v_i) # Tránh lỗi nếu v toàn 0

        # Công thức: Chi tiêu đúng tỷ lệ alpha
        # x_i = (alpha_i * Budget) / Price_i
        x = (alpha * B_i) / safe_price

    # =========================================================
    # 3. LEONTIEF: U = min(x / v)
    # =========================================================
    elif utility_type == "Leontief":
        # Chiến thuật: Mua theo tỷ lệ cố định của v
        # Giá của 1 combo chuẩn = sum(v_i * p_i)
        cost_of_one_bundle = np.dot(v_i, safe_price)

        if cost_of_one_bundle > 0:
            # Số lượng combo mua được
            num_bundles = B_i / cost_of_one_bundle
            x = num_bundles * v_i
        else:
            x = np.zeros(m_goods)

    # =========================================================
    # 4. CES UTILITY: U = (sum v * x^rho)^(1/rho)
    # =========================================================
    elif utility_type == "CES_8":
        # CẤU HÌNH rho (m) TẠI ĐÂY
        # Trong code cũ bạn để m = 1/2
        rho = 0.5

        # --- CASE A: CONCAVE (rho < 1, rho != 0) ---
        # Đây là trường hợp thay thế (Substitute) -> Mua nhiều loại
        if rho < 1:
            # Tính Sigma (Elasticity of Substitution)
            # sigma = 1 / (1 - rho)
            sigma = 1.0 / (1.0 - rho)

            # Tính phần tử tỷ lệ: Term_i = (v_i / p_i)^sigma
            # Dùng np.maximum cho v_i để tránh v=0 gây lỗi log hoặc mũ âm
            term = np.power(v_i / safe_price, sigma)

            # Tính mẫu số chung: Sum (p_j * term_j)
            denom = np.dot(safe_price, term)

            if denom > 0:
                # x_i = (B * term_i) / denom
                x = (B_i * term) / denom
            else:
                 # Fallback nếu v=0 hết
                 x = np.zeros(m_goods)

        # --- CASE B: CONVEX (rho > 1) ---
        # Đây là trường hợp "Winner Takes All" giống Linear
        else:
            # So sánh tỷ lệ: v^(1/rho) / p
            v_transformed = np.power(v_i, 1.0/rho)
            bang_per_buck = v_transformed / safe_price

            best_idx = np.argmax(bang_per_buck)
            x[best_idx] = B_i / safe_price[best_idx]

    return x

def solve_centralized_optimal_fair(valuations, budgets, supply_s, b_mat, A, utility_type="Linear"):
    """
    Returns:
        optimal_value (float): Giá trị hàm mục tiêu tối ưu.
        optimal_X (np.ndarray): Ma trận phân bổ tối ưu (N x M).
    """
    n, m = valuations.shape
    X = cp.Variable((n, m), nonneg=True)

    constraints = [
        cp.sum(X, axis=0) <= supply_s
    ]
    for i in range(n):
        constraints.append(A @ X[i] <= b_mat[i])

    # --- XÂY DỰNG OBJECTIVE ---
    if utility_type == "Linear":
        utilities = cp.sum(cp.multiply(valuations, X), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "Cobb-Douglas":
        eps = 1e-12
        alpha = valuations / (np.sum(valuations, axis=1, keepdims=True))
        log_utilities = cp.sum(cp.multiply(alpha, cp.log(X + eps)), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    elif utility_type == "Leontief":
        # Lưu ý: Leontief trong CVXPY có thể phức tạp/chậm với quy mô lớn
        inv_valuations = np.divide(
            1.0,
            valuations,
            out=np.full_like(valuations, 1e30),
            where=(valuations > 0)
        )

        # 2. Nhân X với ma trận nghịch đảo hằng số này
        # ratio_matrix[i, j] = X[i, j] * (1 / v[i, j])
        weighted_X = cp.multiply(X, inv_valuations)

        # 3. Lấy min theo hàng
        utilities = cp.min(weighted_X, axis=1)

        # 4. Tính hàm mục tiêu
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "CES_8":
        # Nếu chạy Solver tập trung, m NÊN nhỏ hơn hoặc bằng 1 (ví dụ 0.5 hoặc -1).
        # Nếu đặt m = 8, Solver ECOS/SCS sẽ KHÔNG giải được (báo lỗi DCPError).
        eps = 1e-9
        m_ces = 1/2

        # Công thức: log_util = (1/m) * log( sum( v * x^m ) )
        # Tính tổng trọng số lũy thừa theo hàng (axis=1)
        inner_term = cp.sum(cp.multiply(valuations, cp.power(X + eps, m_ces)), axis=1)

        # Logarit hóa hàm mục tiêu
        log_utilities = (1.0 / m_ces) * cp.log(inner_term)

        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    objective = cp.Maximize(primal_utility)
    prob = cp.Problem(objective, constraints)

    print(f"--- Solving Centralized Fair Problem ({utility_type}) ---")
    try:
        prob.solve(solver=cp.ECOS, verbose=False)
    except:
        prob.solve(solver=cp.SCS, verbose=False)

    # TRẢ VỀ: (Giá trị mục tiêu, Ma trận X tối ưu)
    return prob.value, X.value


def one_sample_sgd_fastlog(
    A: np.ndarray, b: np.ndarray, supply_s: np.ndarray, q0: np.ndarray,
    valuations: np.ndarray, budgets: np.ndarray, p0: np.ndarray,
    lr_p=0.01, lr_q=0.01, num_iters=1000, seed=42, log_freq=10,
    utility_type="Linear"
):
    rng = np.random.default_rng(seed)
    n, m = valuations.shape

    # Copy biến dual
    p = p0.copy()
    q = q0.copy()

    # --- BƯỚC 1: KHỞI TẠO TUYỆT ĐỐI (X0 = 100) ---
    X = np.ones((n, m)) * 10
    util = util = np.zeros(n)
    for j in range(n):
        if utility_type == "Linear":
            util[j] = valuations[j] @ X[j]
        elif utility_type == "Cobb-Douglas":
            util[j] = calculate_cd_utility(X[j], valuations[j])
        elif utility_type == "Leontief":
            util[j] = calculate_leo_utility(X[j], valuations[j])
        elif utility_type == "CES_8":
            util[j] = calculate_ces_utility(X[j], valuations[j], m_ces=0.5)

    # Khởi tạo lịch sử
    obj_hist = []
    real_allocation_hist = []

    # --- BƯỚC 2: LƯU TRẠNG THÁI T=0 (KHI X VẪN LÀ 0) ---
    # Tính Objective tại điểm xuất phát (lúc này chưa ai mua gì)
    term_p = np.sum(p * supply_s)
    term_q = np.sum(q * b)
    # Lưu ý: log(0 + 1e-12) sẽ ra số âm rất lớn (khoảng -27), đây là đúng toán học
    term_u = np.sum(budgets * np.log(util + 1e-12))
    obj_0 = term_p + term_q + term_u - np.sum(budgets)

    # LƯU NGAY LẬP TỨC
    obj_hist.append(obj_0)
    real_allocation_hist.append(X.copy()) # Lưu ma trận toàn số 0

    print(f"--- Start SGD ({num_iters} iterations) | Initial Obj (X=0): {obj_0:.2f} ---")
    t0 = time.perf_counter()

    # --- BƯỚC 3: WARM UP (LÀM NÓNG) ---
    # Bây giờ mới tính toán phân bổ lần đầu dựa trên giá p0, q0
    # Bước này cần thiết để có Tổng Cầu (D) khởi tạo cho vòng lặp SGD
    for j in range(n):
        # Dùng hàm Analytical siêu tốc
        X[j] = buyer_best_response_cvx(valuations[j], p, A, q[j], budgets[j], utility_type)

        # Cập nhật util để dùng cho vòng lặp sau
        if utility_type == "Linear":
            util[j] = valuations[j] @ X[j]
        elif utility_type == "Cobb-Douglas":
            util[j] = calculate_cd_utility(X[j], valuations[j])
        elif utility_type == "Leontief":
            util[j] = calculate_leo_utility(X[j], valuations[j])
        elif utility_type == "CES_8":
            util[j] = calculate_ces_utility(X[j], valuations[j], m_ces=0.5)

    # Tính Tổng cầu ban đầu (D) sau khi Warm up
    D = X.sum(axis=0)

    # --- BƯỚC 4: VÒNG LẶP SGD CHÍNH (TỪ T=1) ---
    for t in range(1, num_iters + 1):
        # A. Chọn ngẫu nhiên 1 user
        i = rng.integers(0, n)

        # B. Cập nhật Demand (Incremental)
        D -= X[i]

        # C. Tìm Best Response (Analytical)
        x_new = buyer_best_response_cvx(valuations[i], p, A, q[i], budgets[i], utility_type)

        # Cập nhật X và Utility
        X[i] = x_new

        if utility_type == "Linear":
            util[i] = valuations[i] @ X[i]
        elif utility_type == "Cobb-Douglas":
            util[i] = calculate_cd_utility(X[i], valuations[i])
        elif utility_type == "Leontief":
            util[i] = calculate_leo_utility(X[i], valuations[i])
        elif utility_type == "CES_8":
            util[i] = calculate_ces_utility(X[i], valuations[i], m_ces=0.5)

        D += x_new

        # D. Tính Gradient & Cập nhật Dual
        g_p = D - supply_s
        load_i = A @ x_new
        g_q_i = load_i - b[i]

        alpha = lr_p / np.sqrt(t)
        beta  = lr_q / np.sqrt(t)

        p = np.maximum(0, p + alpha * g_p)
        q[i] = np.maximum(0, q[i] + beta * g_q_i)

        # E. Ghi Log định kỳ
        if t % log_freq == 0:
            term_p = np.sum(p * supply_s)
            term_q = np.sum(q * b)
            term_u = np.sum(budgets * np.log(util + 1e-12))
            obj = term_p + term_q + term_u - np.sum(budgets)

            obj_hist.append(obj)
            real_allocation_hist.append(X.copy())

            if t % 5000 == 0:
                print(f"Iter {t}/{num_iters} | Obj: {obj:.4f}")

    total_time = time.perf_counter() - t0
    print(f"--- Finished in {total_time:.2f}s ---")

    return obj_hist, real_allocation_hist

# ==============================================================================
# 2. MAIN SCRIPT
# ==============================================================================

# --- A. LOAD DATA & SETUP ---
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_10u_24i"
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path): valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
except:
    valuations = np.random.rand(10, 24) + 0.1

N_USERS, M_GOODS = valuations.shape
T_SLOTS = 4

# --- B. PARAMETERS (FIXED SEED) ---
np.random.seed(1)
# [1] Budgets
budgets = np.ones(N_USERS) * 10
supply_s = np.random.uniform(5, 10, M_GOODS)
capacity_b = np.random.uniform(1, 4, (N_USERS, T_SLOTS))

if M_GOODS % T_SLOTS == 0: A_matrix = build_A_matrix(M_GOODS, T_SLOTS)
else: A_matrix = np.zeros((T_SLOTS, M_GOODS))

# --- C. TÌM TARGET (SOLVER) ---
target_val = solve_centralized_optimal_fair(valuations, budgets, supply_s, capacity_b, A_matrix, utility_type="Linear")
print(f"Target V* = {target_val:.4f}")

# --- D. CHẠY THÍ NGHIỆM VỚI 3 STEP SIZES ---
# Định nghĩa 3 mức Learning Rate (Eta)
step_sizes = [0.025, 0.02, 0.005]
labels     = ['High ($\eta=0.025$)', 'Medium ($\eta=0.02$)', 'Low ($\eta=0.005$)']
colors     = ['orange', 'green', 'purple'] # Màu giống hình mẫu

results = {}
LOG_FREQ = 10
NUM_ITERS = 1000000 # Chạy 2000 bước để nhìn rõ

print("\n--- Running Experiments ---")
for lr in step_sizes:
    print(f"Running SGD with lr={lr}...")
    p0 = np.ones(M_GOODS)
    q0 = np.zeros((N_USERS, T_SLOTS))

    hist = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        lr_p=lr, lr_q=lr, # Gán Step Size
        num_iters=NUM_ITERS, log_freq=LOG_FREQ
    )
    save_dir = "/content/drive/MyDrive/EV_charging_project/experiment_results/Fig4"

    # Tạo thư mục nếu chưa tồn tại
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f"Đã tạo thư mục: {save_dir}")

    # Lưu obj_hist vào file pickle
    file_path = os.path.join(save_dir, f"lr_{lr}.pkl")

    with open(file_path, 'wb') as f:
        pickle.dump(hist, f)

    print(f"✅ Đã lưu thành công obj_hist tại: {file_path}")
    results[lr] = hist

<>:344: SyntaxWarning: invalid escape sequence '\e'
<>:344: SyntaxWarning: invalid escape sequence '\e'
<>:344: SyntaxWarning: invalid escape sequence '\e'
<>:344: SyntaxWarning: invalid escape sequence '\e'
<>:344: SyntaxWarning: invalid escape sequence '\e'
<>:344: SyntaxWarning: invalid escape sequence '\e'
/tmp/ipython-input-3166391408.py:344: SyntaxWarning: invalid escape sequence '\e'
  labels     = ['High ($\eta=0.025$)', 'Medium ($\eta=0.02$)', 'Low ($\eta=0.005$)']
/tmp/ipython-input-3166391408.py:344: SyntaxWarning: invalid escape sequence '\e'
  labels     = ['High ($\eta=0.025$)', 'Medium ($\eta=0.02$)', 'Low ($\eta=0.005$)']
/tmp/ipython-input-3166391408.py:344: SyntaxWarning: invalid escape sequence '\e'
  labels     = ['High ($\eta=0.025$)', 'Medium ($\eta=0.02$)', 'Low ($\eta=0.005$)']


AttributeError: module 'cvxpy' has no attribute 'promote'

In [ ]:
# ==============================================================================
# 3. VẼ BIỂU ĐỒ (SENSITIVITY ANALYSIS)
# ==============================================================================
plt.figure(figsize=(10, 6))

iterations = np.arange(len(results[step_sizes[0]])) * LOG_FREQ

# 1. Vẽ 3 đường SGD
for i, lr in enumerate(step_sizes):
    plt.plot(iterations, results[lr], color=colors[i], label=labels[i], linewidth=1.5)

# 2. Vẽ đường Solver
plt.axhline(y=target_val, color='black', linestyle='--', linewidth=2, label='Optimal V* (Solver)')

# 3. Trang trí
plt.xlabel('Iterations (k)', fontsize=12)
plt.ylabel('Objective Function Value', fontsize=12)
plt.title(r'Sensitivity to Step Size ($\eta$) - Objective Minimization', fontsize=14, fontweight='bold')
plt.grid(True, linestyle='-', alpha=0.5) # Grid nhạt giống hình mẫu
plt.legend(loc='upper right', framealpha=1.0) # Legend có khung nền trắng


plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import os
import time

# ==============================================================================
# 1. CÁC HÀM CỐT LÕI (CORE FUNCTIONS)
# ==============================================================================

def build_A_matrix(n_goods, n_slots):
    A = np.zeros((n_slots, n_goods))
    for j in range(n_goods):
        A[j % n_slots, j] = 1.0
    return A

def buyer_best_response_cvx(v_i, p, A, q_i, B_i):
    """Giải bài toán con"""
    m = len(v_i)
    x = cp.Variable(m, nonneg=True)
    effective_price = p + (A.T @ q_i)

    objective = cp.Maximize(B_i * cp.log(v_i @ x + 1e-12))
    constraint = [effective_price @ x <= B_i]

    prob = cp.Problem(objective, constraint)
    try: prob.solve(solver=cp.ECOS)
    except:
        try: prob.solve(solver=cp.SCS)
        except: return np.zeros(m)

    if x.value is None: return np.zeros(m)
    return np.maximum(x.value, 0.0)

def solve_centralized_optimal_fair(valuations, budgets, supply_s, b_mat, A):
    """Solver tìm V* (Dual Target = Primal + Budget)"""
    n, m = valuations.shape
    X = cp.Variable((n, m), nonneg=True)

    utilities = cp.sum(cp.multiply(valuations, X), axis=1)
    primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))
    total_budget_const = np.sum(budgets)

    objective = cp.Maximize(primal_utility + total_budget_const)

    constraints = [cp.sum(X, axis=0) <= supply_s]
    for i in range(n):
        constraints.append(A @ X[i] <= b_mat[i])

    prob = cp.Problem(objective, constraints)
    print(f"--- Solving Solver Target... ---")
    try: prob.solve(solver=cp.ECOS, verbose=False)
    except: prob.solve(solver=cp.SCS, verbose=False)
    return prob.value

def one_sample_sgd_fastlog(
    A, b, supply_s, q0, valuations, budgets, p0,
    lr_p=0.01, lr_q=0.01, num_iters=1000, seed=42, log_freq=10
):
    """SGD trả về lịch sử Objective"""
    rng = np.random.default_rng(seed)
    n, m = valuations.shape

    p = p0.astype(float).copy()
    q = q0.astype(float).copy()

    # Warm start
    X = np.zeros((n, m))
    util = np.zeros(n)
    for j in range(n):
        X[j] = buyer_best_response_cvx(valuations[j], p, A, q[j], budgets[j])
        util[j] = valuations[j] @ X[j]
    D = X.sum(axis=0)

    obj_hist = []

    # Ghi log t=0
    term_p = np.sum(p * supply_s)
    term_q = np.sum(q * b)
    term_u = np.sum(budgets * np.log(util + 1e-12))
    obj_hist.append(term_p + term_q + term_u)

    # Loop
    for t in range(1, num_iters + 1):
        i = int(rng.integers(n))

        D -= X[i]
        x_new = buyer_best_response_cvx(valuations[i], p, A, q[i], budgets[i])
        X[i] = x_new
        util[i] = valuations[i] @ x_new
        D += x_new

        g_p = D - supply_s
        load_i = A @ x_new
        g_q_i = load_i - b[i]

        # Decay Learning Rate
        alpha = lr_p / np.sqrt(t)
        beta  = lr_q / np.sqrt(t)

        p = np.maximum(1e-3, p + alpha * g_p)
        q[i] = np.maximum(0, q[i] + beta * g_q_i)

        if t % log_freq == 0:
            term_p = np.sum(p * supply_s)
            term_q = np.sum(q * b)
            term_u = np.sum(budgets * np.log(util + 1e-12))
            obj_val = term_p + term_q + term_u
            obj_hist.append(term_p + term_q + term_u)
            if t % 500 == 0:
                print(f"Iter {t}/{num_iters} | Obj: {obj_val:.4f}")

    return obj_hist

# ==============================================================================
# 2. MAIN SCRIPT
# ==============================================================================

# --- A. LOAD DATA & SETUP ---
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_10u_24i"
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path): valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
except:
    valuations = np.random.rand(10, 24) + 0.1

N_USERS, M_GOODS = valuations.shape
T_SLOTS = 4

# --- B. PARAMETERS (FIXED SEED) ---
np.random.seed(10)
budgets = np.random.uniform(5.0, 20.0, N_USERS)
supply_s = np.random.uniform(5, 10, M_GOODS)
capacity_b = np.random.uniform(1, 4, (N_USERS, T_SLOTS))

if M_GOODS % T_SLOTS == 0: A_matrix = build_A_matrix(M_GOODS, T_SLOTS)
else: A_matrix = np.zeros((T_SLOTS, M_GOODS))

# --- C. TÌM TARGET (SOLVER) ---
target_val = solve_centralized_optimal_fair(valuations, budgets, supply_s, capacity_b, A_matrix)
print(f"Target V* = {target_val:.4f}")

# --- D. CHẠY THÍ NGHIỆM VỚI 3 STEP SIZES ---
# Định nghĩa 3 mức Learning Rate (Eta)
step_sizes = [0.02, 0.005, 0.001]
labels     = ['High ($\eta=0.02$)', 'Medium ($\eta=0.005$)', 'Low ($\eta=0.001$)']
colors     = ['orange', 'green', 'purple'] # Màu giống hình mẫu

results = {}
LOG_FREQ = 1
NUM_ITERS = 1000000 # Chạy 2000 bước để nhìn rõ

print("\n--- Running Experiments ---")
for lr in step_sizes:
    print(f"Running SGD with lr={lr}...")
    p0 = np.ones(M_GOODS)
    q0 = np.zeros((N_USERS, T_SLOTS))

    hist = one_sample_sgd_fastlog(
        A_matrix, capacity_b, supply_s, q0, valuations, budgets, p0,
        lr_p=lr, lr_q=lr, # Gán Step Size
        num_iters=NUM_ITERS, log_freq=LOG_FREQ, utility_type="Linear"
    )
    save_dir = "/content/drive/MyDrive/EV_charging_project/experiment_results/Fig4_new"

    # Tạo thư mục nếu chưa tồn tại
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        print(f"Đã tạo thư mục: {save_dir}")

    # Lưu obj_hist vào file pickle
    file_path = os.path.join(save_dir, f"lr_{lr}.pkl")

    with open(file_path, 'wb') as f:
        pickle.dump(hist, f)

    print(f"✅ Đã lưu thành công obj_hist tại: {file_path}")
    results[lr] = hist

In [ ]:
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

# ==============================================================================
# 1. CẤU HÌNH & LOAD DỮ LIỆU
# ==============================================================================

# Đường dẫn chứa file
SAVE_DIR = "/content/drive/MyDrive/EV_charging_project/experiment_results/Fig4_new"

# Các tham số cấu hình y hệt lúc chạy
step_sizes = [0.02, 0.005, 0.001]
labels     = [r'High ($\eta=0.02$)', r'Medium ($\eta=0.005$)', r'Low ($\eta=0.001$)']
colors     = ['orange', 'green', 'purple']
LOG_FREQ   = 1

target_val =
results = {}

print("--- Loading Data from Drive ---")
for lr in step_sizes:
    file_name = f"lr_{lr}.pkl"
    file_path = os.path.join(SAVE_DIR, file_name)

    try:
        with open(file_path, 'rb') as f:
            data = pickle.load(f)
            results[lr] = data
            print(f"✅ Loaded: {file_name} (Length: {len(data)})")
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        # Tạo dữ liệu giả để code không crash nếu thiếu file (Optional)
        results[lr] = [target_val] * 150000

# ==============================================================================
# 2. VẼ BIỂU ĐỒ (SENSITIVITY ANALYSIS)
# ==============================================================================
if len(results) > 0:
    plt.figure(figsize=(10, 6))

    # Lấy trục X từ dữ liệu đầu tiên load được
    first_lr = step_sizes[0]
    if first_lr in results:
        iterations = np.arange(len(results[first_lr])) * LOG_FREQ
    else:
        iterations = np.arange(150000) # Fallback

    # 1. Vẽ 3 đường SGD
    for i, lr in enumerate(step_sizes):
        if lr in results:
            # Vẽ raw data để thấy độ dao động (Sensitivity)
            plt.plot(iterations, results[lr],
                     color=colors[i],
                     label=labels[i],
                     linewidth=1.5,
                     alpha=0.9)

    # 2. Vẽ đường Solver Benchmark
    plt.axhline(y=target_val, color='black', linestyle='--', linewidth=2, label='Optimal V* (Solver)')

    # 3. Trang trí
    plt.xlabel('Iterations (k)', fontsize=12)
    plt.ylabel('Objective Function Value', fontsize=12)
    plt.title(r'Sensitivity to Step Size ($\eta$) - Objective Minimization', fontsize=14, fontweight='bold')

    # Grid & Legend đẹp
    plt.grid(True, linestyle='-', alpha=0.5)
    plt.legend(loc='upper right', framealpha=1.0, shadow=True)

    # Giới hạn trục Y (Optional: để nhìn rõ vùng hội tụ hơn nếu biên độ dao động quá lớn)
    # plt.ylim(350, 400)

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Không có dữ liệu nào được load để vẽ hình.")

In [ ]:
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

# ==============================================================================
# 1. CẤU HÌNH & LOAD DỮ LIỆU
# ==============================================================================

SAVE_DIR = "/content/drive/MyDrive/EV_charging_project/experiment_results/Fig4_new"

step_sizes = [0.02, 0.005, 0.001]
labels     = [r'High ($\eta=0.02$)', r'Medium ($\eta=0.005$)', r'Low ($\eta=0.001$)']
colors     = ['orange', 'green', 'purple']
LOG_FREQ   = 1

target_val = 295.0053
results = {}

print("--- Loading Data from Drive ---")
for lr in step_sizes:
    file_name = f"lr_{lr}.pkl"
    file_path = os.path.join(SAVE_DIR, file_name)
    try:
        with open(file_path, 'rb') as f:
            data = pickle.load(f)
            results[lr] = data
            print(f"✅ Loaded: {file_name} (Length: {len(data)})")
    except FileNotFoundError:
        print(f"❌ File not found: {file_path}")
        # Dữ liệu giả để test nếu không tìm thấy file
        results[lr] = [target_val + 100*np.exp(-0.001*i) for i in range(150000)]

# ==============================================================================
# 2. CẤU HÌNH KHOẢNG NHÌN (VIEW RANGE) - CHỈNH Ở ĐÂY
# ==============================================================================

# Bạn muốn xem từ iteration nào đến iteration nào?
START_ITER = 0        # Bắt đầu (Mặc định 0)
END_ITER   = 5000     # Kết thúc (Mặc định None để vẽ đến cuối cùng)
                      # Ví dụ: Để 5000 để xem 5000 vòng lặp đầu tiên

# ==============================================================================
# 3. VẼ BIỂU ĐỒ
# ==============================================================================
if len(results) > 0:
    plt.figure(figsize=(10, 6))

    # Lấy key đầu tiên để tạo trục X chuẩn
    first_lr = step_sizes[0]

    # 1. Tính toán chỉ số cắt (Slicing Indices) dựa trên LOG_FREQ
    # Vì dữ liệu được lưu thưa (mỗi LOG_FREQ bước lưu 1 lần)
    start_idx = int(START_ITER / LOG_FREQ)

    if END_ITER is None:
        end_idx = None # Lấy đến hết
        display_end = "End"
    else:
        end_idx = int(END_ITER / LOG_FREQ)
        display_end = END_ITER

    # Tạo trục X cho khoảng đã chọn
    if first_lr in results:
        full_iterations = np.arange(len(results[first_lr])) * LOG_FREQ
        # Cắt trục X
        iterations_slice = full_iterations[start_idx:end_idx]
    else:
        iterations_slice = []

    # 2. Vẽ 3 đường SGD (Đã cắt theo khoảng)
    for i, lr in enumerate(step_sizes):
        if lr in results:
            # Cắt dữ liệu trục Y tương ứng
            y_data_slice = results[lr][start_idx:end_idx]

            # Kiểm tra độ dài cho khớp (phòng hờ)
            min_len = min(len(iterations_slice), len(y_data_slice))

            plt.plot(iterations_slice[:min_len], y_data_slice[:min_len],
                     color=colors[i],
                     label=labels[i],
                     linewidth=1.5,
                     alpha=0.9)

    # 3. Vẽ đường Solver Benchmark
    plt.axhline(y=target_val, color='black', linestyle='--', linewidth=2, label='Optimal V* (Solver)')

    # 4. Trang trí
    plt.xlabel('Iterations (k)', fontsize=12)
    plt.ylabel('Objective Function Value', fontsize=12)
    plt.title(f'Sensitivity Analysis (View: Iter {START_ITER} to {display_end})', fontsize=14, fontweight='bold')

    plt.grid(True, linestyle='-', alpha=0.5)
    plt.legend(loc='upper right', framealpha=1.0, shadow=True)

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Không có dữ liệu nào được load để vẽ hình.")